# Limpeza e preparação dos dados para PLN

Este notebook filtra jogos e avaliações, limpa os textos e cria representações por tokens para uso posterior em modelos. Execute a coleta primeiro e confirme a existência dos dois CSVs em `_DadosBrutos`.

## Fluxo e arquivos

Execute as células de cima para baixo: instalação é leitura e filtros; limpeza textual; tokenização; normalização; stopwords; stemming e lematização; e comparação. As etapas compartilham DataFrames em memória; após reiniciar o kernel, é necessário refazer as etapas anteriores.

| Diretório | Conteúdo |
| --- | --- |
| `_DadosBrutos/` | Entradas `jogos_steam.csv` e `steam_reviews.csv`, preservadas por este notebook |
| `_DadosLimpos/` | Os mesmos nomes de arquivo, atualizados a cada etapa principal |
| `_DadosLimpos/stemming/` | Versão alternativa com radicais |
| `_DadosLimpos/lematizacao/` | Versão alternativa com lemas |

Os caminhos são relativos à raiz do repositório. Os CSVs de saída são sobrescritos: se a execução parar no meio, os arquivos principais refletem a última etapa salva, e saídas alternativas podem ser de uma execução anterior. Os dados e suas cópias são carregados em memória; as colunas de tokens também aumentam o tamanho dos arquivos.

## 1. Instalação das dependências

`%pip` instala as bibliotecas no ambiente do kernel. Esta etapa pode precisar de acesso é internet. Se o ambiente solicitar reinicialização, reinicie o kernel antes de continuar.


In [1]:
%pip install pandas nltk langdetect simplemma


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/981.5 kB ? eta -:--:--
     ---------------------------------------- 981.5/981.5 kB 7.2 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ---------------------------------------- 1.8/1.8 MB 11.0 MB/s  0:00:00
   ---------------------------------------- 0.0/19.0 MB ? eta -:--:--
   --- ------------------------------------ 1.8/19.0 MB 12.3 MB/s eta 0:00:02
   ------ --------------------------------- 2.9/19.0 MB 7.7 MB/s eta 0:00:03
   -------- ------------------------------- 4.2/19.0 MB 7.4 MB/s eta 0:0


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Leitura e remoção de descrições vazias

Lê os CSVs brutos e identifica jogos com `description` ausente ou composta apenas por espaços. Remove esses jogos e suas avaliações por `appid`, mantendo a coerência desse filtro entre os dois conjuntos.

São necessários, ao longo do notebook, `appid`, `name` e `description` no catálogo e `appid` e `review` nas avaliações. Os primeiros resultados são gravados em `_DadosLimpos`, e as contagens mostram o impacto do filtro.


In [2]:
from pathlib import Path

# Resolve o repositorio a partir da raiz ou de qualquer subpasta.
raiz_projeto = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "1_Coleta_Dados").is_dir()
     and (p / "2_Limpeza_Preparacao").is_dir()),
    None,
)
if raiz_projeto is None:
    raise RuntimeError("Execute o notebook dentro do repositorio PLN_SteamRecommend.")

import pandas as pd


pasta_brutos = raiz_projeto / "_DadosBrutos"
pasta_limpos = raiz_projeto / "_DadosLimpos"
pasta_limpos.mkdir(parents=True, exist_ok=True)

caminho_jogos = pasta_brutos / "jogos_steam.csv"
caminho_reviews = pasta_brutos / "steam_reviews.csv"

jogos = pd.read_csv(caminho_jogos, low_memory=False)
reviews = pd.read_csv(caminho_reviews, low_memory=False)

descricao_vazia = jogos["description"].isna() | jogos["description"].astype("string").str.strip().eq("")
appids_removidos = set(jogos.loc[descricao_vazia, "appid"])

jogos_limpo = jogos.loc[~jogos["appid"].isin(appids_removidos)].copy()
reviews_limpo = reviews.loc[~reviews["appid"].isin(appids_removidos)].copy()

jogos_limpo.to_csv(pasta_limpos / "jogos_steam.csv", index=False)
reviews_limpo.to_csv(pasta_limpos / "steam_reviews.csv", index=False)

print(f"Jogos antes: {len(jogos):,}")
print(f"Reviews antes: {len(reviews):,}")
print(f"Jogos removidos por descricao vazia: {len(appids_removidos):,}")
print(f"Reviews removidas dos jogos sem descricao: {len(reviews) - len(reviews_limpo):,}")
print(f"Jogos depois: {len(jogos_limpo):,}")
print(f"Reviews depois: {len(reviews_limpo):,}")


Jogos antes: 186,406
Reviews antes: 483,431
Jogos removidos por descricao vazia: 344
Reviews removidas dos jogos sem descricao: 1,174
Jogos depois: 186,062
Reviews depois: 482,257


## 3. Filtro de letras não latinas

Remove jogos cujo nome ou descrição contenha qualquer letra não latina, além das avaliações desses jogos. A normalização Unicode NFKC é usada na verificação; acentos latinos, pontuação e símbolos não são motivo de exclusão.

Esse critério verifica o alfabeto, não o idioma: textos em vários idiomas latinos podem permanecer. A verificação desta etapa é aplicada ao nome e é descrição do jogo, não diretamente ao texto das avaliações.


In [3]:
import unicodedata


def tem_letra_nao_latina(texto):
    if pd.isna(texto):
        return False
    # Normaliza variantes tipograficas e preserva acentos, simbolos e pontuacao.
    texto = unicodedata.normalize("NFKC", str(texto))
    return any(
        caractere.isalpha() and "LATIN" not in unicodedata.name(caractere, "")
        for caractere in texto
    )


nome_nao_latino = jogos_limpo["name"].map(tem_letra_nao_latina)
descricao_nao_latina = jogos_limpo["description"].map(tem_letra_nao_latina)
appids_nao_latinos = set(
    jogos_limpo.loc[nome_nao_latino | descricao_nao_latina, "appid"]
)

quantidade_reviews_antes = len(reviews_limpo)
jogos_limpo = jogos_limpo.loc[~jogos_limpo["appid"].isin(appids_nao_latinos)].copy()
reviews_limpo = reviews_limpo.loc[~reviews_limpo["appid"].isin(appids_nao_latinos)].copy()

jogos_limpo.to_csv(pasta_limpos / "jogos_steam.csv", index=False)
reviews_limpo.to_csv(pasta_limpos / "steam_reviews.csv", index=False)

print(f"Jogos removidos por letras nao latinas: {len(appids_nao_latinos):,}")
print(f"Reviews removidas desses jogos: {quantidade_reviews_antes - len(reviews_limpo):,}")
print(f"Jogos restantes: {len(jogos_limpo):,}")
print(f"Reviews restantes: {len(reviews_limpo):,}")


Jogos removidos por letras nao latinas: 12,572
Reviews removidas desses jogos: 15,053
Jogos restantes: 173,490
Reviews restantes: 467,204


## 4. Limpeza dos textos

Extrai texto de HTML, ignora conteúdo de `script` e `style`, normaliza Unicode, remove caracteres invisíveis e de controle e reduz espaços repetidos. Aplica a transformação às descrições e avaliações, preservando acentos, pontuação e negações.

Jogos cuja descrição fica vazia apàs a limpeza e suas avaliações são removidos. Avaliações sem texto são mantidas e contabilizadas. Os CSVs principais são atualizados.


In [4]:
import re
from html.parser import HTMLParser


class ExtratorTextoHTML(HTMLParser):
    blocos = {
        "p", "div", "br", "hr", "li", "ul", "ol", "table", "tr", "td",
        "th", "h1", "h2", "h3", "h4", "h5", "h6", "section", "blockquote",
    }

    def __init__(self):
        super().__init__(convert_charrefs=True)
        self.partes = []
        self.ignorar = 0

    def handle_starttag(self, tag, attrs):
        if tag in {"script", "style"}:
            self.ignorar += 1
        if tag in self.blocos and not self.ignorar:
            self.partes.append(" ")

    def handle_endtag(self, tag):
        if tag in {"script", "style"}:
            self.ignorar = max(0, self.ignorar - 1)
        if tag in self.blocos and not self.ignorar:
            self.partes.append(" ")

    def handle_data(self, data):
        if not self.ignorar:
            self.partes.append(data)


def limpar_texto(texto):
    if pd.isna(texto):
        return ""
    parser = ExtratorTextoHTML()
    parser.feed(str(texto))
    parser.close()
    texto = unicodedata.normalize("NFKC", "".join(parser.partes))
    # Remove caracteres invisiveis sem retirar acentos, pontuacao ou negacoes.
    texto = texto.translate(dict.fromkeys(map(ord, "\u200b\ufeff\u00ad")))
    texto = "".join(
        c for c in texto
        if unicodedata.category(c) != "Cc" or c.isspace()
    )
    return re.sub(r"\s+", " ", texto).strip()


jogos_limpo["description"] = jogos_limpo["description"].map(limpar_texto)
reviews_limpo["review"] = reviews_limpo["review"].map(limpar_texto)

appids_vazios_apos_limpeza = set(
    jogos_limpo.loc[jogos_limpo["description"].eq(""), "appid"]
)
reviews_antes_limpeza = len(reviews_limpo)
jogos_limpo = jogos_limpo.loc[
    ~jogos_limpo["appid"].isin(appids_vazios_apos_limpeza)
].copy()
reviews_limpo = reviews_limpo.loc[
    ~reviews_limpo["appid"].isin(appids_vazios_apos_limpeza)
].copy()

jogos_limpo.to_csv(pasta_limpos / "jogos_steam.csv", index=False)
reviews_limpo.to_csv(pasta_limpos / "steam_reviews.csv", index=False)

print(f"Jogos removidos por descricao vazia apos limpeza: {len(appids_vazios_apos_limpeza):,}")
print(f"Reviews removidas desses jogos: {reviews_antes_limpeza - len(reviews_limpo):,}")
print(f"Reviews sem texto (mantidas): {reviews_limpo['review'].eq('').sum():,}")
print(f"Jogos restantes: {len(jogos_limpo):,}")
print(f"Reviews restantes: {len(reviews_limpo):,}")


Jogos removidos por descricao vazia apos limpeza: 0
Reviews removidas desses jogos: 0
Reviews sem texto (mantidas): 2,250
Jogos restantes: 173,490
Reviews restantes: 467,204


## 5. Tokenização

Usa `TweetTokenizer` para separar os textos em tokens, preservando maiúsculas, repetições de caracteres e identificadores com `@`. Cria `description_tokens` e `review_tokens`.

As listas ficam como listas Python nos DataFrames, mas são serializadas em JSON nos CSVs. Ao reler uma coluna de tokens, use `json.loads` para recuperar cada lista, por exemplo: `df["review_tokens"].map(json.loads)`.


In [5]:
import json
from nltk.tokenize import TweetTokenizer

tokenizador = TweetTokenizer(preserve_case=True, reduce_len=False, strip_handles=False)


def tokenizar_texto(texto):
    if pd.isna(texto):
        return []
    return tokenizador.tokenize(str(texto))


jogos_limpo["description_tokens"] = jogos_limpo["description"].map(tokenizar_texto)
reviews_limpo["review_tokens"] = reviews_limpo["review"].map(tokenizar_texto)

# JSON permite recuperar as listas com json.loads ao reler os CSVs.
for dados, coluna_tokens, nome_arquivo in [
    (jogos_limpo, "description_tokens", "jogos_steam.csv"),
    (reviews_limpo, "review_tokens", "steam_reviews.csv"),
]:
    dados.assign(**{
        coluna_tokens: dados[coluna_tokens].map(
            lambda tokens: json.dumps(tokens, ensure_ascii=False)
        )
    }).to_csv(pasta_limpos / nome_arquivo, index=False)

print(f"Tokens nas descricoes: {jogos_limpo['description_tokens'].map(len).sum():,}")
print(f"Tokens nas reviews: {reviews_limpo['review_tokens'].map(len).sum():,}")
print(f"Reviews sem tokens: {reviews_limpo['review_tokens'].map(len).eq(0).sum():,}")

display(jogos_limpo[["appid", "description", "description_tokens"]].head(3))
display(reviews_limpo[["appid", "review", "review_tokens"]].head(3))


Tokens nas descricoes: 6,819,289
Tokens nas reviews: 46,959,875
Reviews sem tokens: 2,250


,appid,description,description_tokens
0,10,Play the world's number 1 online action game. ...,"[Play, the, world's, number, 1, online, action..."
1,20,One of the most popular online action games of...,"[One, of, the, most, popular, online, action, ..."
2,30,Enlist in an intense brand of Axis vs. Allied ...,"[Enlist, in, an, intense, brand, of, Axis, vs,..."


,appid,review,review_tokens
0,10,"Good game, most perfect counter strike ever ex...","[Good, game, ,, most, perfect, counter, strike..."
1,10,wwww,[wwww]
2,10,This game is good for anyone who wants a smoot...,"[This, game, is, good, for, anyone, who, wants..."


## 6. Normalização dos tokens

Padroniza Unicode e variantes de apóstrofos e converte os tokens para minúsculas. Mantém as colunas anteriores e acrescenta `description_tokens_normalizados` e `review_tokens_normalizados`.

A gravação serializa ambas as versões de tokens em JSON, permitindo comparar a tokenização original com a representação normalizada.


In [6]:
import json
import unicodedata

apostrofos = str.maketrans({"\u2018": "'", "\u2019": "'", "\u02bc": "'"})


def normalizar_tokens(tokens):
    return [
        unicodedata.normalize(
            "NFKC", unicodedata.normalize("NFKC", token).translate(apostrofos).lower()
        )
        for token in tokens
    ]


jogos_limpo["description_tokens_normalizados"] = jogos_limpo["description_tokens"].map(
    normalizar_tokens
)
reviews_limpo["review_tokens_normalizados"] = reviews_limpo["review_tokens"].map(
    normalizar_tokens
)

# Serializa ambas as listas em JSON, mantendo listas nos DataFrames em memoria.
for dados, coluna_tokens, nome_arquivo in [
    (jogos_limpo, "description_tokens", "jogos_steam.csv"),
    (reviews_limpo, "review_tokens", "steam_reviews.csv"),
]:
    colunas_listas = [coluna_tokens, f"{coluna_tokens}_normalizados"]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_limpos / nome_arquivo, index=False)

print(f"Descricoes normalizadas: {len(jogos_limpo):,}")
print(f"Reviews normalizadas: {len(reviews_limpo):,}")

display(jogos_limpo[
    ["appid", "description_tokens", "description_tokens_normalizados"]
].head(3))
display(reviews_limpo[
    ["appid", "review_tokens", "review_tokens_normalizados"]
].head(3))


Descricoes normalizadas: 173,490
Reviews normalizadas: 467,204


,appid,description_tokens,description_tokens_normalizados
0,10,"[Play, the, world's, number, 1, online, action...","[play, the, world's, number, 1, online, action..."
1,20,"[One, of, the, most, popular, online, action, ...","[one, of, the, most, popular, online, action, ..."
2,30,"[Enlist, in, an, intense, brand, of, Axis, vs,...","[enlist, in, an, intense, brand, of, axis, vs,..."


,appid,review_tokens,review_tokens_normalizados
0,10,"[Good, game, ,, most, perfect, counter, strike...","[good, game, ,, most, perfect, counter, strike..."
1,10,[wwww],[wwww]
2,10,"[This, game, is, good, for, anyone, who, wants...","[this, game, is, good, for, anyone, who, wants..."


## 7. Detecção de idioma e remoção de stopwords

Detecta o idioma de cada texto usando até 1.500 caracteres. Textos com menos de cinco palavras ou menos de 20 letras ficam como `indeterminado`; a classificação também exige probabilidade estimada de pelo menos 0,90. A semente fixa torna a detecção reproduzível no mesmo ambiente, mas não garante sua exatidão.

Usa listas do NLTK para inglês, português, espanhol, francês, italiano, alemão e holandês. O recurso `stopwords` é baixado se estiver ausente. Negações são preservadas para reduzir alterações de sentido. Textos sem idioma suportado mantém seus tokens sem filtragem.

Acrescenta as colunas com sufixos `_idioma` e `_tokens_sem_stopwords` e atualiza os CSVs principais.


In [7]:
import nltk
from nltk.corpus import stopwords
from langdetect import DetectorFactory, detect_langs, LangDetectException

try:
    stopwords.fileids()
except LookupError:
    nltk.download("stopwords", raise_on_error=True)

DetectorFactory.seed = 0
idiomas_stopwords = {
    "en": "english", "pt": "portuguese", "es": "spanish",
    "fr": "french", "it": "italian", "de": "german", "nl": "dutch",
}
# Preserva negacoes para nao inverter o sentido, especialmente nas reviews.
negacoes = set(normalizar_tokens([
    "no", "not", "nor", "never", "neither", "nothing", "nobody", "without",
    "cannot", "n't", "n\u00e3o", "nem", "nunca", "jamais", "sem",
    "nada", "ningu\u00e9m", "ninguem", "nao", "ni", "sin", "nadie",
    "ning\u00fan", "ninguno", "ninguna", "tampoco",
    "ne", "n'", "pas", "plus", "rien", "aucun", "aucune", "sans",
    "non", "mai", "nessuno", "nessuna", "niente", "senza",
    "nicht", "kein", "keine", "keinen", "keinem", "keiner", "keines",
    "nie", "niemals", "niemand", "nichts", "ohne",
    "niet", "geen", "nooit", "niemand", "niets", "zonder",
]))
stopwords_por_idioma = {
    codigo: {
        palavra for palavra in normalizar_tokens(stopwords.words(nome))
        if palavra not in negacoes and not palavra.endswith("n't")
    }
    for codigo, nome in idiomas_stopwords.items()
}


def detectar_idioma(texto):
    if pd.isna(texto):
        return "indeterminado"
    # A detecao e heuristica; textos curtos e resultados incertos ficam intactos.
    amostra = str(texto)[:1500]
    if len(amostra.split()) < 5 or sum(c.isalpha() for c in amostra) < 20:
        return "indeterminado"
    try:
        candidato = detect_langs(amostra)[0]
    except LangDetectException:
        return "indeterminado"
    return candidato.lang if candidato.prob >= 0.90 else "indeterminado"


def remover_stopwords(tokens, idioma):
    palavras = stopwords_por_idioma.get(idioma, set())
    return [token for token in tokens if token not in palavras]


for dados, texto, nome_arquivo in [
    (jogos_limpo, "description", "jogos_steam.csv"),
    (reviews_limpo, "review", "steam_reviews.csv"),
]:
    coluna_idioma = f"{texto}_idioma"
    coluna_entrada = f"{texto}_tokens_normalizados"
    coluna_saida = f"{texto}_tokens_sem_stopwords"
    dados[coluna_idioma] = dados[texto].map(detectar_idioma)
    dados[coluna_saida] = [
        remover_stopwords(tokens, idioma)
        for tokens, idioma in zip(dados[coluna_entrada], dados[coluna_idioma])
    ]

    colunas_listas = [f"{texto}_tokens", coluna_entrada, coluna_saida]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_limpos / nome_arquivo, index=False)

    removidos = dados[coluna_entrada].map(len).sum() - dados[coluna_saida].map(len).sum()
    sem_lista = ~dados[coluna_idioma].isin(stopwords_por_idioma)
    print(f"{texto}: {removidos:,} stopwords removidas")
    print(f"{texto}: {sem_lista.sum():,} textos mantidos sem filtragem por idioma")
    display(dados[["appid", coluna_idioma, coluna_entrada, coluna_saida]].head(3))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\berna\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


description: 2,316,699 stopwords removidas
description: 2,125 textos mantidos sem filtragem por idioma


,appid,description_idioma,description_tokens_normalizados,description_tokens_sem_stopwords
0,10,en,"[play, the, world's, number, 1, online, action...","[play, world's, number, 1, online, action, gam..."
1,20,en,"[one, of, the, most, popular, online, action, ...","[one, popular, online, action, games, time, ,,..."
2,30,en,"[enlist, in, an, intense, brand, of, axis, vs,...","[enlist, intense, brand, axis, vs, ., allied, ..."


review: 17,717,535 stopwords removidas
review: 100,063 textos mantidos sem filtragem por idioma


,appid,review_idioma,review_tokens_normalizados,review_tokens_sem_stopwords
0,10,en,"[good, game, ,, most, perfect, counter, strike...","[good, game, ,, perfect, counter, strike, ever..."
1,10,indeterminado,[wwww],[wwww]
2,10,en,"[this, game, is, good, for, anyone, who, wants...","[game, good, anyone, wants, smooth, ,, casual,..."


## 8. Alternativa A: stemming

Parte de cópias dos dados apàs a remoção de stopwords e aplica `SnowballStemmer` por idioma. Stemming produz radicais, que podem não ser palavras completas. Negações, tokens não alfabéticos e idiomas sem stemmer permanecem intactos.

Acrescenta colunas com sufixo `_tokens_stemming` e salva os dois CSVs em `_DadosLimpos/stemming/`. Esta alternativa não substitui os DataFrames da etapa principal.


In [8]:
import json
from functools import lru_cache
from nltk.stem.snowball import SnowballStemmer

stemmers = {
    codigo: SnowballStemmer(nome)
    for codigo, nome in idiomas_stopwords.items()
    if nome in SnowballStemmer.languages
}


@lru_cache(maxsize=100000)
def aplicar_stemming(token, idioma):
    stemmer = stemmers.get(idioma)
    if stemmer is None or token in negacoes or not token.isalpha():
        return token
    return stemmer.stem(token)


# As duas alternativas partem dos tokens sem stopwords.
jogos_stemming = jogos_limpo.copy()
reviews_stemming = reviews_limpo.copy()
pasta_stemming = pasta_limpos / "stemming"
pasta_stemming.mkdir(parents=True, exist_ok=True)

for dados, texto, nome_arquivo in [
    (jogos_stemming, "description", "jogos_steam.csv"),
    (reviews_stemming, "review", "steam_reviews.csv"),
]:
    entrada = f"{texto}_tokens_sem_stopwords"
    saida = f"{texto}_tokens_stemming"
    idiomas = dados[f"{texto}_idioma"]
    dados[saida] = [
        [aplicar_stemming(token, idioma) for token in tokens]
        for tokens, idioma in zip(dados[entrada], idiomas)
    ]
    colunas_listas = [
        f"{texto}_tokens", f"{texto}_tokens_normalizados", entrada, saida,
    ]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_stemming / nome_arquivo, index=False)
    print(f"{texto}: {len(dados):,} registros salvos em {pasta_stemming / nome_arquivo}")
    print(f"Idioma sem stemmer (tokens mantidos): {(~idiomas.isin(stemmers)).sum():,}")
    display(dados[["appid", f"{texto}_idioma", entrada, saida]].head(3))


description: 173,490 registros salvos em c:\Users\berna\Documents\GitHub\PLN_SteamRecommend\DadosLimpos\stemming\jogos_steam.csv
Idioma sem stemmer (tokens mantidos): 2,125


,appid,description_idioma,description_tokens_sem_stopwords,description_tokens_stemming
0,10,en,"[play, world's, number, 1, online, action, gam...","[play, world's, number, 1, onlin, action, game..."
1,20,en,"[one, popular, online, action, games, time, ,,...","[one, popular, onlin, action, game, time, ,, t..."
2,30,en,"[enlist, intense, brand, axis, vs, ., allied, ...","[enlist, intens, brand, axi, vs, ., alli, team..."


review: 467,204 registros salvos em c:\Users\berna\Documents\GitHub\PLN_SteamRecommend\DadosLimpos\stemming\steam_reviews.csv
Idioma sem stemmer (tokens mantidos): 100,063


,appid,review_idioma,review_tokens_sem_stopwords,review_tokens_stemming
0,10,en,"[good, game, ,, perfect, counter, strike, ever...","[good, game, ,, perfect, counter, strike, ever..."
1,10,indeterminado,[wwww],[wwww]
2,10,en,"[game, good, anyone, wants, smooth, ,, casual,...","[game, good, anyon, want, smooth, ,, casual, c..."


## 9. Alternativa B: lematização

Parte dos mesmos tokens sem stopwords usados no stemming, sem aplicar lematização aos radicais. Usa `simplemma` para obter lemas por dicionário nos idiomas configurados; não resolve ambiguidades pelo contexto da frase.

Negações, tokens não alfabéticos e idiomas não suportados permanecem intactos. Acrescenta colunas com sufixo `_tokens_lematizados` e salva os dois CSVs em `_DadosLimpos/lematizacao/`.


In [9]:
import json
from functools import lru_cache
import simplemma

idiomas_lematizacao = {"en", "pt", "es", "fr", "it", "de", "nl"}


@lru_cache(maxsize=100000)
def lematizar_token(token, idioma):
    if idioma not in idiomas_lematizacao or token in negacoes or not token.isalpha():
        return token
    # Lematizacao por dicionario: nao desambigua pelo contexto da frase.
    return simplemma.lemmatize(token, lang=idioma)


jogos_lematizados = jogos_limpo.copy()
reviews_lematizadas = reviews_limpo.copy()
pasta_lematizacao = pasta_limpos / "lematizacao"
pasta_lematizacao.mkdir(parents=True, exist_ok=True)

for dados, texto, nome_arquivo in [
    (jogos_lematizados, "description", "jogos_steam.csv"),
    (reviews_lematizadas, "review", "steam_reviews.csv"),
]:
    entrada = f"{texto}_tokens_sem_stopwords"
    saida = f"{texto}_tokens_lematizados"
    idiomas = dados[f"{texto}_idioma"]
    dados[saida] = [
        [lematizar_token(token, idioma) for token in tokens]
        for tokens, idioma in zip(dados[entrada], idiomas)
    ]
    colunas_listas = [
        f"{texto}_tokens", f"{texto}_tokens_normalizados", entrada, saida,
    ]
    dados.assign(**{
        coluna: dados[coluna].map(lambda tokens: json.dumps(tokens, ensure_ascii=False))
        for coluna in colunas_listas
    }).to_csv(pasta_lematizacao / nome_arquivo, index=False)
    print(f"{texto}: {len(dados):,} registros salvos em {pasta_lematizacao / nome_arquivo}")
    print(f"Idioma sem lematizador (tokens mantidos): {(~idiomas.isin(idiomas_lematizacao)).sum():,}")
    display(dados[["appid", f"{texto}_idioma", entrada, saida]].head(3))


description: 173,490 registros salvos em c:\Users\berna\Documents\GitHub\PLN_SteamRecommend\DadosLimpos\lematizacao\jogos_steam.csv
Idioma sem lematizador (tokens mantidos): 2,125


,appid,description_idioma,description_tokens_sem_stopwords,description_tokens_lematizados
0,10,en,"[play, world's, number, 1, online, action, gam...","[play, world's, number, 1, online, action, gam..."
1,20,en,"[one, popular, online, action, games, time, ,,...","[one, popular, online, action, game, time, ,, ..."
2,30,en,"[enlist, intense, brand, axis, vs, ., allied, ...","[enlist, intense, brand, axis, versus, ., ally..."


review: 467,204 registros salvos em c:\Users\berna\Documents\GitHub\PLN_SteamRecommend\DadosLimpos\lematizacao\steam_reviews.csv
Idioma sem lematizador (tokens mantidos): 100,063


,appid,review_idioma,review_tokens_sem_stopwords,review_tokens_lematizados
0,10,en,"[good, game, ,, perfect, counter, strike, ever...","[good, game, ,, perfect, counter, strike, ever..."
1,10,indeterminado,[wwww],[wwww]
2,10,en,"[game, good, anyone, wants, smooth, ,, casual,...","[game, good, anyone, want, smooth, ,, casual, ..."


## 10. Comparação das etapas

Seleciona até 10 jogos mantidos pela limpeza e a primeira avaliação disponível de cada um. Jogos sem avaliação continuam na amostra. Mostra o texto original, a limpeza, o idioma, os tokens e as duas alternativas finais.

Esta célula é demonstrativa e não grava arquivos. A exibição limita textos e listas para facilitar a leitura; os resultados completos ficam nos DataFrames. Use a comparação para inspecionar transformações, sem tratá-la como avaliação quantitativa da qualidade do pipeline. As saídas já armazenadas no notebook podem corresponder a execuções anteriores.


In [10]:
from IPython.display import display

# Seleciona ate 10 jogos mantidos e a primeira review disponivel de cada um.
ids_pipeline = jogos_limpo["appid"].drop_duplicates().head(10)
amostra_jogos = jogos.loc[
    jogos["appid"].isin(ids_pipeline), ["appid", "name", "description"]
].drop_duplicates("appid")
amostra_reviews = (
    reviews.loc[reviews["appid"].isin(ids_pipeline), ["appid", "review"]]
    .drop_duplicates("appid", keep="first")
    .assign(tem_review=True)
)
amostra_pipeline = (
    amostra_jogos
    .merge(amostra_reviews, on="appid", how="left", validate="one_to_one")
    .assign(tem_review=lambda df: df["tem_review"].eq(True))
    .rename(columns={"description": "descricao_original", "review": "review_original"})
)


def etapa_filtros(df):
    return df.assign(
        descricao_valida=~(
            df["descricao_original"].isna()
            | df["descricao_original"].astype("string").str.strip().eq("")
        ),
        letras_latinas=~(
            df["name"].map(tem_letra_nao_latina)
            | df["descricao_original"].map(tem_letra_nao_latina)
        ),
    )


def etapa_textos(df, sufixo_entrada, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": df[f"{campo}_{sufixo_entrada}"].map(funcao)
        for campo in ("descricao", "review")
    })


def etapa_por_idioma(df, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": [
            funcao(tokens, idioma)
            for tokens, idioma in zip(
                df[f"{campo}_normalizados"], df[f"{campo}_idioma"]
            )
        ]
        for campo in ("descricao", "review")
    })


def etapa_alternativa(df, sufixo_saida, funcao):
    return df.assign(**{
        f"{campo}_{sufixo_saida}": [
            [funcao(token, idioma) for token in tokens]
            for tokens, idioma in zip(
                df[f"{campo}_sem_stopwords"], df[f"{campo}_idioma"]
            )
        ]
        for campo in ("descricao", "review")
    })


resultado_pipeline = (
    amostra_pipeline
    .pipe(etapa_filtros)
    .pipe(etapa_textos, "original", "limpa", limpar_texto)
    .pipe(etapa_textos, "limpa", "idioma", detectar_idioma)
    .pipe(etapa_textos, "limpa", "tokens", tokenizar_texto)
    .pipe(etapa_textos, "tokens", "normalizados", normalizar_tokens)
    .pipe(etapa_por_idioma, "sem_stopwords", remover_stopwords)
    .pipe(etapa_alternativa, "stemming", aplicar_stemming)
    .pipe(etapa_alternativa, "lematizados", lematizar_token)
)

etapas_pipeline = {
    "Texto original": "original",
    "Limpeza": "limpa",
    "Idioma estimado": "idioma",
    "Tokenizacao": "tokens",
    "Normalizacao": "normalizados",
    "Sem stopwords": "sem_stopwords",
    "Stemming (alternativa)": "stemming",
    "Lematizacao (alternativa)": "lematizados",
}
comparacao_pipeline = pd.concat([
    resultado_pipeline[
        ["appid", "name", f"descricao_{sufixo}", f"review_{sufixo}"]
    ].rename(columns={
        f"descricao_{sufixo}": "descricao",
        f"review_{sufixo}": "review",
    }).assign(etapa=etapa)
    for etapa, sufixo in etapas_pipeline.items()
], ignore_index=True)

print(f"Pipeline demonstrativo: {len(resultado_pipeline)} jogos mantidos pela limpeza.")
print("Reviews: primeira linha de cada jogo; jogos sem review permanecem na amostra.")
print("Previa limitada a 160 caracteres/20 tokens; resultados completos nos DataFrames.")
with pd.option_context("display.max_colwidth", 160, "display.max_seq_items", 20):
    display(resultado_pipeline[
        ["appid", "name", "descricao_valida", "letras_latinas", "tem_review"]
    ])
    for appid, nome in resultado_pipeline[["appid", "name"]].itertuples(index=False, name=None):
        print(f"{appid} - {nome}")
        display(
            comparacao_pipeline.loc[comparacao_pipeline["appid"].eq(appid)]
            .set_index("etapa")[["descricao", "review"]]
        )


Pipeline demonstrativo: 10 jogos mantidos pela limpeza.
Reviews: primeira linha de cada jogo; jogos sem review permanecem na amostra.
Previa limitada a 160 caracteres/20 tokens; resultados completos nos DataFrames.


,appid,name,descricao_valida,letras_latinas,tem_review
0,10,Counter-Strike,True,True,True
1,20,Team Fortress Classic,True,True,True
2,30,Day of Defeat,True,True,True
3,40,Deathmatch Classic,True,True,True
4,50,Half-Life: Opposing Force,True,True,True
5,60,Ricochet,True,True,True
6,70,Half-Life,True,True,True
7,80,Counter-Strike: Condition Zero,True,True,True
8,130,Half-Life: Blue Shift,True,True,True
9,220,Half-Life 2,True,True,True


10 - Counter-Strike


,descricao,review
etapa,,
Texto original,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with...,"Good game, most perfect counter strike ever existed the only problem is this version does not have bots only Condition Zero has bots"
Limpeza,Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with...,"Good game, most perfect counter strike ever existed the only problem is this version does not have bots only Condition Zero has bots"
Idioma estimado,en,en
Tokenizacao,"[Play, the, world's, number, 1, online, action, game, ., Engage, in, an, incredibly, realistic, brand, of, terrorist, warfare, in, this, ...]","[Good, game, ,, most, perfect, counter, strike, ever, existed, the, only, problem, is, this, version, does, not, have, bots, only, ...]"
Normalizacao,"[play, the, world's, number, 1, online, action, game, ., engage, in, an, incredibly, realistic, brand, of, terrorist, warfare, in, this, ...]","[good, game, ,, most, perfect, counter, strike, ever, existed, the, only, problem, is, this, version, does, not, have, bots, only, ...]"
Sem stopwords,"[play, world's, number, 1, online, action, game, ., engage, incredibly, realistic, brand, terrorist, warfare, wildly, popular, team-based, game, ., ally, ...]","[good, game, ,, perfect, counter, strike, ever, existed, problem, version, not, bots, condition, zero, bots]"
Stemming (alternativa),"[play, world's, number, 1, onlin, action, game, ., engag, incred, realist, brand, terrorist, warfar, wild, popular, team-based, game, ., alli, ...]","[good, game, ,, perfect, counter, strike, ever, exist, problem, version, not, bot, condit, zero, bot]"
Lematizacao (alternativa),"[play, world's, number, 1, online, action, game, ., engage, incredibly, realistic, brand, terrorist, warfare, wildly, popular, team-based, game, ., ally, ...]","[good, game, ,, perfect, counter, strike, ever, exist, problem, version, not, bot, condition, zero, bot]"


20 - Team Fortress Classic


...

,descricao,review
etapa,,
Texto original,"With its extensive Tour of Duty campaign, a near-limitless number of skirmish modes, updates and new content for Counter-Strike's award-winning multiplayer ...",i love this game... NOT!!!!!
Limpeza,"With its extensive Tour of Duty campaign, a near-limitless number of skirmish modes, updates and new content for Counter-Strike's award-winning multiplayer ...",i love this game... NOT!!!!!
Idioma estimado,en,indeterminado
Tokenizacao,"[With, its, extensive, Tour, of, Duty, campaign, ,, a, near-limitless, number, of, skirmish, modes, ,, updates, and, new, content, for, ...]","[i, love, this, game, ..., NOT, !, !, !]"
Normalizacao,"[with, its, extensive, tour, of, duty, campaign, ,, a, near-limitless, number, of, skirmish, modes, ,, updates, and, new, content, for, ...]","[i, love, this, game, ..., not, !, !, !]"
Sem stopwords,"[extensive, tour, duty, campaign, ,, near-limitless, number, skirmish, modes, ,, updates, new, content, counter-strike's, award-winning, multiplayer, game, ...","[i, love, this, game, ..., not, !, !, !]"
Stemming (alternativa),"[extens, tour, duti, campaign, ,, near-limitless, number, skirmish, mode, ,, updat, new, content, counter-strike's, award-winning, multiplay, game, play, ,,...","[i, love, this, game, ..., not, !, !, !]"
Lematizacao (alternativa),"[extensive, tour, duty, campaign, ,, near-limitless, number, skirmish, mode, ,, update, new, content, counter-strike's, award-winning, multiplayer, game, pl...","[i, love, this, game, ..., not, !, !, !]"


130 - Half-Life: Blue Shift


,descricao,review
etapa,,
Texto original,"Made by Gearbox Software and originally released in 2001 as an add-on to Half-Life, Blue Shift is a return to the Black Mesa Research Facility in which you ...",blueshit
Limpeza,"Made by Gearbox Software and originally released in 2001 as an add-on to Half-Life, Blue Shift is a return to the Black Mesa Research Facility in which you ...",blueshit
Idioma estimado,en,indeterminado
Tokenizacao,"[Made, by, Gearbox, Software, and, originally, released, in, 2001, as, an, add-on, to, Half-Life, ,, Blue, Shift, is, a, return, ...]",[blueshit]
Normalizacao,"[made, by, gearbox, software, and, originally, released, in, 2001, as, an, add-on, to, half-life, ,, blue, shift, is, a, return, ...]",[blueshit]
Sem stopwords,"[made, gearbox, software, originally, released, 2001, add-on, half-life, ,, blue, shift, return, black, mesa, research, facility, play, barney, calhoun, ,, ...",[blueshit]
Stemming (alternativa),"[made, gearbox, softwar, origin, releas, 2001, add-on, half-life, ,, blue, shift, return, black, mesa, research, facil, play, barney, calhoun, ,, ...]",[blueshit]
Lematizacao (alternativa),"[make, gearbox, software, originally, release, 2001, add-on, half-life, ,, blue, shift, return, black, mesa, research, facility, play, barney, calhoun, ,, ...]",[blueshit]


220 - Half-Life 2


,descricao,review
etapa,,
Texto original,"Reawakened from stasis in the occupied metropolis of City 17, Gordon Freeman is joined by Alyx Vance as he leads a desperate human resistance. Experience th...",epic
Limpeza,"Reawakened from stasis in the occupied metropolis of City 17, Gordon Freeman is joined by Alyx Vance as he leads a desperate human resistance. Experience th...",epic
Idioma estimado,en,indeterminado
Tokenizacao,"[Reawakened, from, stasis, in, the, occupied, metropolis, of, City, 17, ,, Gordon, Freeman, is, joined, by, Alyx, Vance, as, he, ...]",[epic]
Normalizacao,"[reawakened, from, stasis, in, the, occupied, metropolis, of, city, 17, ,, gordon, freeman, is, joined, by, alyx, vance, as, he, ...]",[epic]
Sem stopwords,"[reawakened, stasis, occupied, metropolis, city, 17, ,, gordon, freeman, joined, alyx, vance, leads, desperate, human, resistance, ., experience, landmark, ...",[epic]
Stemming (alternativa),"[reawaken, stasi, occupi, metropoli, citi, 17, ,, gordon, freeman, join, alyx, vanc, lead, desper, human, resist, ., experi, landmark, first-person, ...]",[epic]
Lematizacao (alternativa),"[reawaken, stasis, occupy, metropolis, city, 17, ,, Gordon, freeman, join, alyx, vance, lead, desperate, human, resistance, ., experience, landmark, first-p...",[epic]
